# Settings

In [4]:
import os
import glob
from pympi.Elan import Eaf
import cv2
import numpy as np
import pandas as pd

curfolder = os.getcwd()


# Cut videos to trial-sized clips

## Cut videos based on ELAN annos (IP)

In [3]:
videofolder = os.path.join(curfolder, 'data\\IP\\')
elanfolder = os.path.join(curfolder, 'data\\IP\\IP EAF\\')
outputfolder = os.path.join(curfolder, 'data\\IP\\IP_trials\\')

videofiles = glob.glob(videofolder + '*.mp4')
elanfiles = glob.glob(elanfolder + '*.eaf')

tier = 'Sign/gesture'


In [4]:
# make a dictionary of the lags for each video_name

vidnames = []
for video in videofiles:
    vidname = video.split('\\')[-1].split('.')[0]
    # keep only first two elements
    vidname = '_'.join(vidname.split('_')[:2])
    vidnames.append(vidname)

# now make a dictionary of the lags for each vid_name, put there 0 for now
vidlags = {vidname: 0 for vidname in vidnames}


In [5]:
# IP06_02 has lag of 250 ms
vidlags['IP06_02'] = 250
vidlags['IP08_02'] = 1050
vidlags['IP10_02'] = 550
vidlags['IP16_02'] = -50

In [6]:
for video in videofiles:

    # Get the name of the video file
    video_name = os.path.basename(video).split('.')[0]
    # keep only first two elements of the name
    video_name = video_name.split('_')[0] + '_' + video_name.split('_')[1]

    # Get the corresponding ELAN file
    elanfile = [f for f in elanfiles if video_name in f][0]
    print(video_name)
    print(elanfile)

    # Load the ELAN file
    eaf = Eaf(elanfile)

    # Get the annotations from the tier
    annotations = eaf.get_annotation_data_for_tier(tier)

    # Get the video
    cap = cv2.VideoCapture(video)
    # Get the frame rate
    fps = cap.get(cv2.CAP_PROP_FPS)
    # get the length of the video by time
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps)
    print('length of the video: ' + str(length))
    # number of frames
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print('number of frames: ' + str(frames))
    
    #Process the annotations
    for anno in annotations:
        print('annotation: ' + anno[2])
        start_time = anno[0] / 1000  # Convert ms to seconds
        end_time = anno[1] / 1000  # Convert ms to seconds
        annotation_value = anno[2]

        # Calculate start and end frames
        start_frame = int(start_time * fps)
        end_frame = int(end_time * fps)

        frames = range(start_frame, end_frame)

        # Set the video to the start frame
        #cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        # if annotation value contains Ä, Ö, Ü, ß, replace it with AE, OE, UE, SS (capital)
        annotation_value = annotation_value.replace('Ä', 'AE')
        annotation_value = annotation_value.replace('Ö', 'OE')
        annotation_value = annotation_value.replace('Ü', 'UE')
        annotation_value = annotation_value.replace('ß', 'SS')


        # Set output filename
        outputfilename = os.path.join(outputfolder, video_name + '_' + annotation_value + '.mp4')

        # check if the video already exists
        if os.path.exists(outputfilename):
            print('video already exists: ' + outputfilename)
            continue

        # Define videowriter
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        out = cv2.VideoWriter(outputfilename, fourcc, fps, (width, height))

        for fra in frames:
            # add 100 frames because we are not sure if the frames are loaded correctly
            # get lag from the dictionary
            fra_lag = fra + vidlags[video_name]
            cap.set(cv2.CAP_PROP_POS_FRAMES, fra_lag)
            ret, frame = cap.read()
            if ret:
                # display th loading image with im.show() to see if the frames are loaded correctly
                #cv2.imshow('frame',frame)
                #cv2.waitKey(1000)
                out.write(frame)
                
            if not ret:
                print('a frame was dropped: ' + str(fra))

        
            
        out.release()

    cap.release()
    cv2.destroyAllWindows()

print('done')

    

IP06_02
e:\VICOM_SignGes_Trajectory\data\IP\IP EAF\IP06_02_DS.eaf
length of the video: 1907
number of frames: 47676
annotation: SCHMETTERLING
video already exists: e:\VICOM_SignGes_Trajectory\data\IP\IP_trials\IP06_02_SCHMETTERLING.mp4
annotation: WARTEN
video already exists: e:\VICOM_SignGes_Trajectory\data\IP\IP_trials\IP06_02_WARTEN.mp4
annotation: PFANNE
video already exists: e:\VICOM_SignGes_Trajectory\data\IP\IP_trials\IP06_02_PFANNE.mp4
annotation: KÄMMEN
video already exists: e:\VICOM_SignGes_Trajectory\data\IP\IP_trials\IP06_02_KAEMMEN.mp4
annotation: TEUFEL
video already exists: e:\VICOM_SignGes_Trajectory\data\IP\IP_trials\IP06_02_TEUFEL.mp4
annotation: ZAUN
video already exists: e:\VICOM_SignGes_Trajectory\data\IP\IP_trials\IP06_02_ZAUN.mp4
annotation: WEINEN
video already exists: e:\VICOM_SignGes_Trajectory\data\IP\IP_trials\IP06_02_WEINEN.mp4
annotation: TASCHENRECHNER
video already exists: e:\VICOM_SignGes_Trajectory\data\IP\IP_trials\IP06_02_TASCHENRECHNER.mp4
annotatio

## Cut videos based on csv (LP1)

In [4]:
import numpy as np
import pandas as pd

videofolder = os.path.join(curfolder, 'data\\LP\\')
outputfolder = os.path.join(videofolder, 'LP_trials\\')

videofiles = glob.glob(videofolder + '*.mov')
# keep only those that have _01
videofiles = [v for v in videofiles if '_01' in v]
csvfiles = glob.glob(videofolder + '*.csv')



In [8]:
# ADD VIDLAG

In [5]:
for video in videofiles:

    print(video)

    if 'LP01' in video or 'LP04' in video or 'LP17' in video or 'LP14' in video or 'LP22' in video or 'LP25' in video:
        # we solve this later
        continue

    else:

        # Get the name of the video file
        video_name = os.path.basename(video).split('.')[0]
        # keep only first two elements of the name
        video_name = video_name.split('_')[0] + '_' + video_name.split('_')[1]

        # Get the corresponding csv file
        csvfile = [file for file in csvfiles if video_name in file][0]

        # Load the ELAN file
        df = pd.read_csv(csvfile)

        # get item_DE, SignStart, SignEnd
        subdf = df[['item_DE', 'SignStart', 'SignEnd']]

        print(subdf)

        # Get the video
        cap = cv2.VideoCapture(video)
        # Get the frame rate
        fps = cap.get(cv2.CAP_PROP_FPS)
        # get the length of the video by time
        length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps)
        print('length of the video: ' + str(length))
        # number of frames
        frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print('number of frames: ' + str(frames))
        
        # loop through the subdf
        for index, row in subdf.iterrows():
            # print what item_DE is being processed
            print(row['item_DE'])
            # get the start and end, convert to seconds (it's in mm:ss:ms)
            start_time = row['SignStart']
            # convert the value to int format
            start_time = int(start_time.split(':')[1]) * 60 + int(start_time.split(':')[2].split('.')[0]) + int(start_time.split(':')[2].split('.')[1]) / 1000
            end_time = row['SignEnd']
            end_time = int(end_time.split(':')[1]) * 60 + int(end_time.split(':')[2].split('.')[0]) + int(end_time.split(':')[2].split('.')[1]) / 1000
            annotation_value = row['item_DE']

            # Calculate start and end frames
            start_frame = int(start_time * fps)
            end_frame = int(end_time * fps)

            frames = range(start_frame, end_frame)

            print(frames)

            # Set the video to the start frame
            #cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

            # if annotation value contains Ä, Ö, Ü, ß, replace it with AE, OE, UE, SS (capital)
            annotation_value = annotation_value.replace('Ä', 'AE')
            annotation_value = annotation_value.replace('Ö', 'OE')
            annotation_value = annotation_value.replace('Ü', 'UE')
            annotation_value = annotation_value.replace('ß', 'SS')


            # Set output filename
            outputfilename = os.path.join(outputfolder, video_name + '_' + annotation_value + '.mp4')

            # check if the video already exists
            if os.path.exists(outputfilename):
                print('video already exists: ' + outputfilename)
                continue

            # Define videowriter
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            
            out = cv2.VideoWriter(outputfilename, fourcc, fps, (width, height))

            for fra in frames:
                # add 100 frames because we are not sure if the frames are loaded correctly
                # get lag from the dictionary
                fra_lag = fra #+ vidlags[video_name]
                cap.set(cv2.CAP_PROP_POS_FRAMES, fra_lag)
                ret, frame = cap.read()
                if ret:
                    # display th loading image with im.show() to see if the frames are loaded correctly
                    #cv2.imshow('frame',frame)
                    #cv2.waitKey(1000)
                    out.write(frame)
                    
                if not ret:
                    print('a frame was dropped: ' + str(fra))

            
                
            out.release()

        cap.release()
        cv2.destroyAllWindows()

    print('done')

    

e:\VICOM_SignGes_Trajectory\data\LP\LP07_01.mov
           item_DE     SignStart       SignEnd
0            FRAGE  00:03:27.429  00:03:31.429
1            MILCH  00:03:37.284  00:03:41.284
2         PFLÜCKEN  00:03:46.982  00:03:50.982
3          DUSCHEN  00:03:57.303  00:04:01.303
4             BABY  00:04:07.320  00:04:11.320
..             ...           ...           ...
132       PYRAMIDE  00:26:10.162  00:26:14.162
133  ARGUMENTIEREN  00:26:19.961  00:26:23.961
134       RASIEREN  00:26:30.200  00:26:34.200
135         KÄMMEN  00:26:40.475  00:26:44.475
136           KALT  00:26:49.790  00:26:53.790

[137 rows x 3 columns]
length of the video: 1716
number of frames: 51489
FRAGE
range(6222, 6342)
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP07_01_FRAGE.mp4
MILCH
range(6518, 6638)
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP07_01_MILCH.mp4
PFLÜCKEN
range(6809, 6929)
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials

## Cut LP2

In [13]:
import openpyxl

videofolder = os.path.join(curfolder, 'data\\LP\\')
outputfolder = os.path.join(videofolder, 'LP_trials\\')

videofiles = glob.glob(videofolder + '*.mov')
# pick only those that have _02 in the name
videofiles = [v for v in videofiles if '_02' in v]
csvfiles = glob.glob(videofolder + '*.csv')
# keep only SignRepetition
csvfiles = [c for c in csvfiles if 'SignRepetition' in c]
 
# read in excel file Offset LP
offsets = pd.read_excel(os.path.join(videofolder, 'Offset LP.xlsx'))

In [14]:
for video in videofiles:

    if 'LP01' in video or 'LP17' in video or 'LP33' in video or 'LP34' in video or 'LP36' in video or 'LP41' in video or 'LP42' in video:
        continue

    # Get the name of the video file
    video_name = os.path.basename(video).split('.')[0]
    # keep only first two elements of the name
    video_name = video_name.split('_')[0] + '_' + video_name.split('_')[1]

    # Get the corresponding csv file
    csvfile = [file for file in csvfiles if video_name in file][0]

    # Load the psychopy file
    df = pd.read_csv(csvfile)

    # get item_DE, SignStart, SignEnd
    subdf = df[['item_DE', 'Repeat_txt.started']]
    # discard first three rows
    subdf = subdf.iloc[3:]
    # discard rows where item_DE is NaN
    subdf = subdf.dropna(subset=['item_DE'])


    # Get the video
    cap = cv2.VideoCapture(video)
    # Get the frame rate
    fps = cap.get(cv2.CAP_PROP_FPS)
    # get the length of the video by time
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps)
    print('length of the video: ' + str(length))
    # number of frames
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print('number of frames: ' + str(frames))
    
    # loop through the subdf
    for index, row in subdf.iterrows():
        # print what item_DE is being processed
        print(row['item_DE'])
        # get the start and end, convert to seconds (it's in mm:ss:ms)
        start_time = row['Repeat_txt.started']
        # in oddawta, get the row where first column has value video_name
        offrow = offsets[offsets.iloc[:,0] == video_name]
        # get the offset from the coumn Offset in this row
        offset = offrow['Offset']
        print('offset: ' + str(offset))
        start_time = start_time + offset
        # end is start + 4 seconds
        end_time = start_time + 4
        annotation_value = row['item_DE']

        # Calculate start and end frames
        start_frame = int(start_time * fps)
        end_frame = int(end_time * fps)

        frames = range(start_frame, end_frame)

        # Set the video to the start frame
        #cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        # if annotation value contains Ä, Ö, Ü, ß, replace it with AE, OE, UE, SS (capital)
        annotation_value = annotation_value.replace('Ä', 'AE')
        annotation_value = annotation_value.replace('Ö', 'OE')
        annotation_value = annotation_value.replace('Ü', 'UE')
        annotation_value = annotation_value.replace('ß', 'SS')


        # Set output filename
        outputfilename = os.path.join(outputfolder, video_name + '_' + annotation_value + '.mp4')

        # check if the video already exists
        if os.path.exists(outputfilename):
            print('video already exists: ' + outputfilename)
            continue

        # Define videowriter
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        out = cv2.VideoWriter(outputfilename, fourcc, fps, (width, height))

        for fra in frames:
            # add 100 frames because we are not sure if the frames are loaded correctly
            # get lag from the dictionary
            fra_lag = fra #+ vidlags[video_name]
            cap.set(cv2.CAP_PROP_POS_FRAMES, fra_lag)
            ret, frame = cap.read()
            if ret:
                # display th loading image with im.show() to see if the frames are loaded correctly
                #cv2.imshow('frame',frame)
                #cv2.waitKey(1000)
                out.write(frame)
                
            if not ret:
                print('a frame was dropped: ' + str(fra))

        
            
        out.release()

    cap.release()
    cv2.destroyAllWindows()

print('done')

    

length of the video: 1523
number of frames: 45705
ZEIT
offset: 1   -47.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP07_02_ZEIT.mp4
TISCHTENNIS
offset: 1   -47.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP07_02_TISCHTENNIS.mp4
RENNEN
offset: 1   -47.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP07_02_RENNEN.mp4
KOTZEN
offset: 1   -47.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP07_02_KOTZEN.mp4
KÖRPER WASCHEN
offset: 1   -47.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP07_02_KOERPER WASCHEN.mp4
TELEFONANRUF
offset: 1   -47.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP07_02_TELEFONANRUF.mp4
SEIFE
offset: 1   -47.0
Name: Offset, dtype: float64
video already ex

C:\Users\Sarka Kadava\AppData\Local\Temp\ipykernel_4656\2253444072.py:53: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  start_frame = int(start_time * fps)
C:\Users\Sarka Kadava\AppData\Local\Temp\ipykernel_4656\2253444072.py:54: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  end_frame = int(end_time * fps)


video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP07_02_UMZIEHEN.mp4
TRINKEN
offset: 1   -47.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP07_02_TRINKEN.mp4
KLAUEN
offset: 1   -47.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP07_02_KLAUEN.mp4
VIDEOSPIEL
offset: 1   -47.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP07_02_VIDEOSPIEL.mp4
WOHNEN
offset: 1   -47.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP07_02_WOHNEN.mp4
SÄGE
offset: 1   -47.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP07_02_SAEGE.mp4
TEE
offset: 1   -47.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP07_02_TEE.mp4
HOCKEY
offset: 1   -47.0
Name: Offset, dtype: float64
video alre

C:\Users\Sarka Kadava\AppData\Local\Temp\ipykernel_4656\2253444072.py:53: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  start_frame = int(start_time * fps)
C:\Users\Sarka Kadava\AppData\Local\Temp\ipykernel_4656\2253444072.py:54: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  end_frame = int(end_time * fps)


video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP13_02_VORHANG .mp4
SEIFE
offset: 2    16.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP13_02_SEIFE.mp4
FOTOGRAFIE
offset: 2    16.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP13_02_FOTOGRAFIE.mp4
TASCHENRECHNER
offset: 2    16.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP13_02_TASCHENRECHNER.mp4
BEZAHLEN
offset: 2    16.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP13_02_BEZAHLEN.mp4
PFANNE
offset: 2    16.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP13_02_PFANNE.mp4
KLEIDUNG
offset: 2    16.0
Name: Offset, dtype: float64
video already exists: e:\VICOM_SignGes_Trajectory\data\LP\LP_trials\LP13_02_KLEIDUNG.mp4
PFEFFER
offset: 2    16.0
Name: Off

C:\Users\Sarka Kadava\AppData\Local\Temp\ipykernel_4656\2253444072.py:53: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  start_frame = int(start_time * fps)
C:\Users\Sarka Kadava\AppData\Local\Temp\ipykernel_4656\2253444072.py:54: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  end_frame = int(end_time * fps)


... (5 outputs left out)

C:\Users\Sarka Kadava\AppData\Local\Temp\ipykernel_4656\2253444072.py:53: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  start_frame = int(start_time * fps)
C:\Users\Sarka Kadava\AppData\Local\Temp\ipykernel_4656\2253444072.py:54: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  end_frame = int(end_time * fps)


LIPPENSTIFT AUFTRAGEN
offset: 10   -3102.0
Name: Offset, dtype: float64
WEINEN
offset: 10   -3102.0
Name: Offset, dtype: float64
ARGUMENTIEREN
offset: 10   -3102.0
Name: Offset, dtype: float64
VORHANG 
offset: 10   -3102.0
Name: Offset, dtype: float64
STECKER
offset: 10   -3102.0
Name: Offset, dtype: float64
VERGANGENHEIT
offset: 10   -3102.0
Name: Offset, dtype: float64
RAUCHEN
offset: 10   -3102.0
Name: Offset, dtype: float64
BIER
offset: 10   -3102.0
Name: Offset, dtype: float64
VIEL
offset: 10   -3102.0
Name: Offset, dtype: float64
HAMMER
offset: 10   -3102.0
Name: Offset, dtype: float64
HÄNDE
offset: 10   -3102.0
Name: Offset, dtype: float64
TRINKWASSER
offset: 10   -3102.0
Name: Offset, dtype: float64
KÖRPER WASCHEN
offset: 10   -3102.0
Name: Offset, dtype: float64
KOTZEN
offset: 10   -3102.0
Name: Offset, dtype: float64
SÄGE
offset: 10   -3102.0
Name: Offset, dtype: float64
AKKORDEON
offset: 10   -3102.0
Name: Offset, dtype: float64
VERHEIRATET
offset: 10   -3102.0
Name: Offset,

C:\Users\Sarka Kadava\AppData\Local\Temp\ipykernel_4656\2253444072.py:53: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  start_frame = int(start_time * fps)
C:\Users\Sarka Kadava\AppData\Local\Temp\ipykernel_4656\2253444072.py:54: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  end_frame = int(end_time * fps)


KARTEN
offset: 13   -478.0
Name: Offset, dtype: float64
HAARE
offset: 13   -478.0
Name: Offset, dtype: float64
VERGANGENHEIT
offset: 13   -478.0
Name: Offset, dtype: float64
ZEIT
offset: 13   -478.0
Name: Offset, dtype: float64
TRETEN
offset: 13   -478.0
Name: Offset, dtype: float64
BRIEF
offset: 13   -478.0
Name: Offset, dtype: float64
BEZAHLEN
offset: 13   -478.0
Name: Offset, dtype: float64
NEUGIERIG
offset: 13   -478.0
Name: Offset, dtype: float64
HOCKEY
offset: 13   -478.0
Name: Offset, dtype: float64
VOGEL
offset: 13   -478.0
Name: Offset, dtype: float64
BRILLE
offset: 13   -478.0
Name: Offset, dtype: float64
AUSSCHALTEN
offset: 13   -478.0
Name: Offset, dtype: float64
BUCH
offset: 13   -478.0
Name: Offset, dtype: float64
BILLIARD
offset: 13   -478.0
Name: Offset, dtype: float64
SCHRAUBENZIEHER
offset: 13   -478.0
Name: Offset, dtype: float64
SOCKEN
offset: 13   -478.0
Name: Offset, dtype: float64
SUPERMARKT
offset: 13   -478.0
Name: Offset, dtype: float64
STINKEN
offset: 13   -4

## Cut gestures

In [5]:
videofolder = os.path.join(curfolder, 'data\\Gestures\\')
outputfolder = os.path.join(videofolder, 'Gestures_trials\\')
csvfolder = os.path.join(videofolder, 'GP_files\\')

videofiles = glob.glob(videofolder + '*.mp4')
csvfiles = glob.glob(csvfolder + '*.csv')

 
# read in excel file Offset LP
offsets = pd.read_excel(os.path.join(csvfolder, 'Offset GP.xlsx'))



In [6]:
## 01 by 150
## 15 by 50
## 16 by 100


for video in videofiles:

    if 'GP01' in video or 'GP15' in video or 'GP16' in video or 'GP03' in video or 'GP04' in video:
        continue

    # Get the name of the video file
    video_name = os.path.basename(video).split('.')[0]
    # keep only first two elements of the name
    video_name = video_name.split('_')[0].split('-')[0]

    # Get the corresponding csv file
    csvfile = [file for file in csvfiles if video_name in file][0]

    # Load the psychopy file
    df = pd.read_csv(csvfile)

    # get item_DE, SignStart, SignEnd
    subdf = df[['text_de', 'text_Prompt.started']]
    # discard rows where item_DE is NaN
    subdf = subdf.dropna(subset=['text_de'])

    # Get the video
    cap = cv2.VideoCapture(video)
    # Get the frame rate
    fps = cap.get(cv2.CAP_PROP_FPS)
    # get the length of the video by time
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps)
    print('length of the video: ' + str(length))
    # number of frames
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print('number of frames: ' + str(frames))
    
    # loop through the subdf
    for index, row in subdf.iterrows():
        # print what item_DE is being processed
        #print(row['text_de'])
        # get the start and end, convert to seconds (it's in mm:ss:ms)
        start_time = row['text_Prompt.started']
        # in oddawta, get the row where first column has value video_name
        offrow = offsets[offsets.iloc[:,0] == video_name]
        # get the offset from the coumn Offset in this row
        offset = offrow['Offset']
        #print('offset: ' + str(offset))
        start_time = start_time + offset
        # end is start + 4 seconds
        end_time = start_time + 4
        annotation_value = row['text_de']
        print(annotation_value)

        # Calculate start and end frames
        start_frame = int(start_time * fps)
        end_frame = int(end_time * fps)

        frames = range(start_frame, end_frame)

        # Set the video to the start frame
        #cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        # if annotation value contains Ä, Ö, Ü, ß, replace it with AE, OE, UE, SS (capital)
        annotation_value = annotation_value.replace('Ä', 'AE')
        annotation_value = annotation_value.replace('Ö', 'OE')
        annotation_value = annotation_value.replace('Ü', 'UE')
        annotation_value = annotation_value.replace('ß', 'SS')


        # Set output filename
        outputfilename = os.path.join(outputfolder, video_name + '_' + annotation_value + '.mp4')

        # check if the video already exists
        if os.path.exists(outputfilename):
            print('video already exists: ' + outputfilename)
            continue

        # Define videowriter
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        out = cv2.VideoWriter(outputfilename, fourcc, fps, (width, height))

        for fra in frames:
            # add 100 frames because we are not sure if the frames are loaded correctly
            # get lag from the dictionary
            fra_lag = fra #+ vidlags[video_name]
            cap.set(cv2.CAP_PROP_POS_FRAMES, fra_lag)
            ret, frame = cap.read()
            if ret:
                # display th loading image with im.show() to see if the frames are loaded correctly
                #cv2.imshow('frame',frame)
                #cv2.waitKey(1000)
                out.write(frame)
                
            if not ret:
                print('a frame was dropped: ' + str(fra))

        
            
        out.release()

    cap.release()
    cv2.destroyAllWindows()

print('done')

    

length of the video: 1718
number of frames: 85900
GESCHENK
video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_GESCHENK.mp4
WEGGEHEN
video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_WEGGEHEN.mp4
KÖNNEN
video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_KOENNEN.mp4
PARKEN
video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_PARKEN.mp4
EINHORN
video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_EINHORN.mp4
SMARTPHONE
video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_SMARTPHONE.mp4
FEUERZEUG
video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_FEUERZEUG.mp4
ANTWORTEN
video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_ANTWORTEN.mp4
TELEFON
video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_TE

C:\Users\Sarka Kadava\AppData\Local\Temp\ipykernel_13832\2635050498.py:56: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  start_frame = int(start_time * fps)
C:\Users\Sarka Kadava\AppData\Local\Temp\ipykernel_13832\2635050498.py:57: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  end_frame = int(end_time * fps)


video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_VIDEO.mp4
SCHICKEN
video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_SCHICKEN.mp4
WOHNEN
video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_WOHNEN.mp4
GAST
video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_GAST.mp4
ADLER
video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_ADLER.mp4
SKI FAHREN
video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_SKI FAHREN.mp4
TOILETTE 
video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_TOILETTE .mp4
WINKEN
video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_WINKEN.mp4
DUSCHEN
video already exists: e:\VICOM_SignGes_Trajectory\data\Gestures\Gestures_trials\GP02_DUSCHEN.mp4
PISTOLE
video already exists: e:\VICOM_SignGes_Trajectory\data\Gest

C:\Users\Sarka Kadava\AppData\Local\Temp\ipykernel_13832\2635050498.py:56: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  start_frame = int(start_time * fps)
C:\Users\Sarka Kadava\AppData\Local\Temp\ipykernel_13832\2635050498.py:57: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  end_frame = int(end_time * fps)


MÖNCH
UMARMEN
ZUSAMMENSTOßEN
DECKE
FUCHS
VIDEOSPIEL
JUNGE
UNTERTITEL
BEOBACHTEN
BISS
FLASCHE
STEHEN
FLIEGEN
LIPPENSTIFT AUFTRAGEN
GRAUSAM
ABLEHNEN
MEDIZIN
MÄDCHEN
PINGUIN
MÜCKE
VIDEO
BODEN
SCHMETTERLING
ANGELN
ERZÄHLEN
GLÜCK
ZUSAMMENHEFTEN
DUFT
KRABBELN
PILZ
BART
FEUER
FRISEUR
DVD
FISCH
AUSWRINGEN
UMZIEHEN
KALENDER
BESTELLEN
ATMEN
GEHÖRLOS
HEMD
HEBEN
ZERREIßEN
VIEL
GABEL
EGAL
ABEND
UNTERGEHEN
LEIHEN
PISTOLE
ORGANISIEREN
MORGEN
ZIEGE
HAHN 
FILM
AUSSCHALTEN
KISSEN
GELD
WASSERSCHILDKRÖTE
SEEPFERDCHEN
DRECK
LANGWEILIG
KÄMMEN
TOILETTE 
DAUMEN DRÜCKEN
ANTWORTEN
PECH
KLAVIER
ERMORDEN
KAMEL
NASHORN
ARBEITEN
BÜCHERREGAL
PUDERN 
VOGEL
FANGEN 
SEE
WOHNEN
KOPIEREN
BETEN
MITGLEID
SCHWEIN
EINFACH
ABHOLEN
WINDRAD
AUSWÄHLEN
RUDERN
STAUB WISCHEN


KeyboardInterrupt: 